# Video J/Z/other con mapas de landmarks

Este experimento sustituye el flujo óptico por una secuencia de mapas RGB dibujados desde los 21 landmarks de MediaPipe. Cada clip genera 17 mapas de 96×96; la normalización usa un origen y escala comunes para todo el clip, preservando la trayectoria entre fotogramas. Un CNN pequeño extrae la postura por fotograma y una GRU aprende el movimiento temporal. Se conserva el split por signante y se guarda en rutas nuevas, sin sobrescribir los otros experimentos.

**Limitación:** la detección de landmarks sigue siendo necesaria. Si falla en más de 5 de los 17 fotogramas, el clip se excluye; los fallos restantes se interpolan. `other` continúa significando los negativos presentes en el dataset, no todas las señas posibles.

In [ ]:
%pip install -q mediapipe==0.10.35 scikit-learn opencv-python-headless
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import csv, hashlib, json, shutil, urllib.request, warnings
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED=42; SEQUENCE_FRAMES=17; MIN_DETECTED_FRAMES=12; FLOW_SIZE=128; FLOW_CLIP_PIXELS=20.0
BATCH_SIZE=8; EPOCHS=35; AUTOTUNE=tf.data.AUTOTUNE
CLASSES=["J","Z","other"]; CLASS_TO_ID={name:i for i,name in enumerate(CLASSES)}
tf.keras.utils.set_random_seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive/MyDrive/proyecto-ia2")
VIDEO_ROOT=DRIVE_ROOT/"data/videos_jz"
WORK_DIR=DRIVE_ROOT/"data/processed/proyecto_def/video_map_sequence_v1"
EXPORT_DIR=DRIVE_ROOT/"exports/proyecto_def/video_map_sequence_v1"
MANIFEST=WORK_DIR/"video_manifest.csv"; SPLIT_FILE=WORK_DIR/"split_map_sequence_v1.csv"
MODEL_ASSET=DRIVE_ROOT/"exports/proyecto_def/hand_landmarker.task"
TEMPORAL_MODEL=EXPORT_DIR/"jz_landmark_map_gru.keras"
WORK_DIR.mkdir(parents=True,exist_ok=True); EXPORT_DIR.mkdir(parents=True,exist_ok=True)
if not VIDEO_ROOT.is_dir(): raise FileNotFoundError(f"No encuentro {VIDEO_ROOT}")
if not MODEL_ASSET.exists():
    MODEL_ASSET.parent.mkdir(parents=True,exist_ok=True)
    url="https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(url,MODEL_ASSET)
print("Videos:",VIDEO_ROOT,"\nSe esperan J/<signer>/clip, Z/<signer>/clip, other/<signer>/clip")
print("Los IDs de participante deben ser globales y repetirse en J y Z para la misma persona.")
print("Procesamiento y checkpoints:",EXPORT_DIR)


## Preprocesar videos a secuencias de mapas

El manifiesto y los mapas se guardan incrementalmente en Drive. Si agregas clips nuevos, vuelve a ejecutar la celda. Si reemplazas un video con el mismo nombre y ruta, cambia la versión de WORK_DIR o elimina su mapa/manifiesto asociado para forzar el reprocesamiento.

In [ ]:
VIDEO_EXTENSIONS={".mp4",".avi",".mov",".mkv"}
MAP_SIZE=96; MIN_DETECTED_FRAMES=12
MAP_FINGERS=[[(0,1),(1,2),(2,3),(3,4)],[(0,5),(5,6),(6,7),(7,8)],
             [(0,9),(9,10),(10,11),(11,12)],[(0,13),(13,14),(14,15),(15,16)],
             [(0,17),(17,18),(18,19),(19,20)]]
MAP_COLORS=[(255,90,90),(255,210,40),(80,235,100),(50,190,255),(220,90,255)]
MAP_PALM_EDGES=[(5,9),(9,13),(13,17)]
MAP_POINT_COLORS=[MAP_COLORS[0]]*4+[MAP_COLORS[1]]*4+[MAP_COLORS[2]]*4+[MAP_COLORS[3]]*4+[MAP_COLORS[4]]*4

def canonical_label(name):
    value=name.strip().lower()
    return {"j":"J","z":"Z","other":"other","none":"other","background":"other"}.get(value)

def sample_frames(path,count=SEQUENCE_FRAMES):
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total>=2:
        indices=np.linspace(0,total-1,count).round().astype(int); frames=[]
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,frame=cap.read()
            if not ok: frames=[]; break
            frames.append(frame)
        cap.release()
        if len(frames)==count: return frames
    else: cap.release()
    # Sequential fallback for videos with missing or inaccurate frame metadata.
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    total=0
    while True:
        ok,_=cap.read()
        if not ok: break
        total+=1
    cap.release()
    if total<2: return None
    wanted=set(np.linspace(0,total-1,count).round().astype(int).tolist())
    selected={}; cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): cap.release(); return None
    i=0
    while True:
        ok,frame=cap.read()
        if not ok: break
        if i in wanted: selected[i]=frame.copy()
        i+=1
    cap.release()
    if not wanted.issubset(selected): return None
    return [selected[int(i)] for i in np.linspace(0,total-1,count).round().astype(int)]

def detect_points(frames,detector):
    seq=[]
    for frame in frames:
        rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        result=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
        if result.hand_landmarks:
            h,w=frame.shape[:2]
            # Use a square-normalized image coordinate system to preserve aspect ratio.
            denom=float(max(h,w))
            seq.append(np.asarray([(p.x*w/denom,p.y*h/denom,p.z) for p in result.hand_landmarks[0]],dtype=np.float32))
        else: seq.append(None)
    detected=sum(p is not None for p in seq)
    if detected<MIN_DETECTED_FRAMES: return None,f"MediaPipe detectó {detected}/{SEQUENCE_FRAMES}; mínimo {MIN_DETECTED_FRAMES}"
    valid=np.asarray([i for i,p in enumerate(seq) if p is not None],dtype=np.float32)
    stacked=np.stack([p for p in seq if p is not None])
    full=np.empty((SEQUENCE_FRAMES,21,3),dtype=np.float32)
    for point_idx in range(21):
        for axis in range(3):
            full[:,point_idx,axis]=np.interp(np.arange(SEQUENCE_FRAMES),valid,stacked[:,point_idx,axis])
    return full,""

def render_map_sequence(points_seq):
    # One clip-level origin and scale preserve fingertip/wrist trajectories across frames.
    wrist=points_seq[:,0,:2]
    center=wrist.mean(axis=0)
    palm=np.linalg.norm(points_seq[:,9,:2]-points_seq[:,0,:2],axis=1)
    scale=max(float(np.median(palm)),1e-5)*3.0
    xy=(points_seq[:,:,:2]-center[None,None,:])/scale+0.5
    xy=np.clip(xy,0.0,1.0)*(MAP_SIZE-1)
    maps=[]
    for frame_points in xy:
        image=np.zeros((MAP_SIZE,MAP_SIZE,3),dtype=np.uint8)
        coords=np.rint(frame_points).astype(np.int32)
        for edges,color in zip(MAP_FINGERS,MAP_COLORS):
            for start,end in edges:
                cv2.line(image,tuple(coords[start]),tuple(coords[end]),color,2,cv2.LINE_AA)
        for start,end in MAP_PALM_EDGES:
            cv2.line(image,tuple(coords[start]),tuple(coords[end]),(235,235,235),2,cv2.LINE_AA)
        for idx,(x,y) in enumerate(coords):
            color=(255,255,255) if idx==0 else MAP_POINT_COLORS[idx-1]
            cv2.circle(image,(int(x),int(y)),2,color,-1,cv2.LINE_AA)
        maps.append(image)
    return np.stack(maps).astype(np.uint8)

def prepare_landmark_maps():
    old={}
    if MANIFEST.exists():
        old_df=pd.read_csv(MANIFEST).drop_duplicates("video_rel",keep="last")
        old={str(r.video_rel):r._asdict() for r in old_df.itertuples(index=False)}
    files=sorted(p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
    if not files: raise ValueError("No hay videos compatibles en VIDEO_ROOT")
    header=not MANIFEST.exists() or MANIFEST.stat().st_size==0
    options=mp.tasks.vision.HandLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,num_hands=1,min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5)
    detector=mp.tasks.vision.HandLandmarker.create_from_options(options); rows=[]
    try:
        with MANIFEST.open("a",newline="",encoding="utf-8") as stream:
            writer=csv.DictWriter(stream,fieldnames=["video_rel","label","signer","map_rel","ok","error"])
            if header: writer.writeheader()
            for i,path in enumerate(files,1):
                rel=path.relative_to(VIDEO_ROOT); parts=rel.parts
                label=canonical_label(parts[0]) if len(parts)>=3 else None
                if label is None: warnings.warn(f"Ignoro ruta; debe ser clase/signante/video: {rel}"); continue
                signer=parts[1]
                map_rel=f"maps/{label}/{signer}/{hashlib.sha1(rel.as_posix().encode()).hexdigest()}.npy"
                output=WORK_DIR/map_rel; previous=old.get(rel.as_posix())
                if previous and output.is_file() and int(previous["ok"])==1:
                    rows.append(previous); continue
                output.parent.mkdir(parents=True,exist_ok=True)
                frames=sample_frames(path)
                if frames is None: sequence,error=None,"video ilegible o con menos de dos fotogramas"
                else:
                    points,error=detect_points(frames,detector)
                    sequence=render_map_sequence(points) if points is not None else None
                ok=sequence is not None
                if ok: np.save(output,sequence)
                record={"video_rel":rel.as_posix(),"label":label,"signer":signer,
                    "map_rel":map_rel,"ok":int(ok),"error":error}
                writer.writerow(record); rows.append(record); stream.flush()
                if i%10==0 or i==len(files): print(f"{i}/{len(files)} | {rel} | {'OK' if ok else error}")
    finally: detector.close()
    return pd.DataFrame(rows).drop_duplicates("video_rel",keep="last")

manifest_df=prepare_landmark_maps()
print("Conteos procesados:")
display(manifest_df[manifest_df.ok.astype(int)==1].groupby(["label","signer"]).size().unstack(fill_value=0))


## Split por signante y entrenamiento

El split busca mantener signantes separados y las tres clases presentes en las particiones. El modelo usa pesos de clase, checkpoints por época y `BackupAndRestore` en Drive para reanudar después de una desconexión.

In [ ]:
usable=manifest_df[manifest_df.ok.astype(int)==1].copy()
if not set(CLASSES).issubset(set(usable.label)):
    raise ValueError(f"Se requieren clips procesables para {CLASSES}; conteos: {usable.label.value_counts().to_dict()}")
if usable.signer.nunique()<3:
    raise ValueError("Se requieren al menos tres IDs de signante; crea carpetas reales por persona.")

if SPLIT_FILE.exists():
    split_df=pd.read_csv(SPLIT_FILE)
    if set(split_df.video_rel)!=set(usable.video_rel):
        raise ValueError("Cambió el conjunto de videos desde el split guardado. Usa split_balanced_v2.csv nuevo o revisa la versión existente.")
else:
    # Asigna signantes completos y busca un reparto cercano a 70/15/15 por clase.
    # Validation y test deben tener al menos cinco clips de cada etiqueta.
    signer_names=sorted(usable.signer.unique())
    group_counts=np.zeros((len(signer_names),len(CLASSES)),dtype=np.int32)
    for gi,signer in enumerate(signer_names):
        counts=usable[usable.signer==signer].label.value_counts()
        for ci,label in enumerate(CLASSES): group_counts[gi,ci]=int(counts.get(label,0))
    fractions=np.asarray([0.70,0.15,0.15],dtype=np.float64)
    totals=group_counts.sum(axis=0).astype(np.float64)
    targets=fractions[:,None]*totals[None,:]
    rng=np.random.default_rng(SEED)
    best_assignment=None; best_score=np.inf
    for _ in range(100000):
        assignment=rng.choice(3,size=len(signer_names),p=fractions)
        counts=np.stack([group_counts[assignment==split].sum(axis=0) for split in range(3)])
        if np.any(counts[0]<1) or np.any(counts[1:]<5): continue
        relative=(counts-targets)/np.maximum(targets,1.0)
        score=float(np.square(relative).sum())
        if score<best_score:
            best_score=score; best_assignment=assignment.copy()
    if best_assignment is None:
        summary={name:{label:int(group_counts[i,j]) for j,label in enumerate(CLASSES) if group_counts[i,j]}
                 for i,name in enumerate(signer_names)}
        raise ValueError("No encontré split por signante con las tres clases y al menos 5 clips "
                         f"por clase en validación/prueba. Signantes y conteos: {summary}")
    split_names=np.asarray(["train","validation","test"])
    signer_to_split={name:split_names[best_assignment[i]] for i,name in enumerate(signer_names)}
    split_df=usable.copy(); split_df["split"]=split_df.signer.map(signer_to_split)
    split_df.to_csv(SPLIT_FILE,index=False)
    print("Error relativo del balance por clase:",round(best_score,4))
display(split_df.groupby(["split","label"]).size().unstack(fill_value=0))
display(split_df.groupby("split").signer.unique().rename("signantes"))

train_df=split_df[split_df.split=="train"].copy()
val_df=split_df[split_df.split=="validation"].copy()
test_df=split_df[split_df.split=="test"].copy()

def stack_maps(frame):
    X=np.stack([np.load(WORK_DIR/p) for p in frame.map_rel])
    y=frame.label.map(CLASS_TO_ID).to_numpy(dtype=np.int32)
    return X,y

X_train,y_train=stack_maps(train_df); X_val,y_val=stack_maps(val_df)
class_weights=compute_class_weight(class_weight="balanced",classes=np.arange(len(CLASSES)),y=y_train)
CLASS_WEIGHTS={i:float(w) for i,w in enumerate(class_weights)}
print("Clips:",{"train":len(train_df),"validation":len(val_df)})
print("Pesos de clase:",CLASS_WEIGHTS)

def build_map_sequence_model():
    inputs=tf.keras.Input((SEQUENCE_FRAMES,MAP_SIZE,MAP_SIZE,3),name="hand_landmark_map_sequence",dtype=tf.uint8)
    x=tf.keras.layers.TimeDistributed(tf.keras.layers.Rescaling(1./255))(inputs)
    for filters in (16,32,64):
        x=tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(filters,3,strides=2,padding="same",use_bias=False))(x)
        x=tf.keras.layers.TimeDistributed(tf.keras.layers.BatchNormalization())(x)
        x=tf.keras.layers.TimeDistributed(tf.keras.layers.Activation("relu"))(x)
    x=tf.keras.layers.TimeDistributed(tf.keras.layers.GlobalAveragePooling2D())(x)
    x=tf.keras.layers.GRU(48,dropout=0.25,return_sequences=False,name="temporal_gru")(x)
    x=tf.keras.layers.Dropout(0.35)(x)
    outputs=tf.keras.layers.Dense(len(CLASSES),activation="softmax",name="class_probs")(x)
    model=tf.keras.Model(inputs,outputs,name="jz_landmark_map_cnn_gru")
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

complete=EXPORT_DIR/"training_complete.json"
if TEMPORAL_MODEL.exists() and complete.exists():
    map_model=tf.keras.models.load_model(TEMPORAL_MODEL,compile=False)
else:
    map_model=build_map_sequence_model()
    ckpt_dir=EXPORT_DIR/"checkpoints"; ckpt_dir.mkdir(parents=True,exist_ok=True)
    callbacks=[
        tf.keras.callbacks.BackupAndRestore(backup_dir=str(EXPORT_DIR/"backup"),save_freq="epoch"),
        tf.keras.callbacks.ModelCheckpoint(str(ckpt_dir/"best.keras"),monitor="val_loss",mode="min",save_best_only=True,verbose=1),
        tf.keras.callbacks.ModelCheckpoint(str(ckpt_dir/"epoch-{epoch:02d}.weights.h5"),save_weights_only=True,save_best_only=False,save_freq="epoch"),
        tf.keras.callbacks.CSVLogger(str(EXPORT_DIR/"history.csv"),append=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=3,min_lr=1e-6),
    ]
    history=map_model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=45,batch_size=8,
        class_weight=CLASS_WEIGHTS,callbacks=callbacks,verbose=2)
    best=ckpt_dir/"best.keras"
    if best.exists(): map_model=tf.keras.models.load_model(best,compile=False)
    map_model.save(TEMPORAL_MODEL)
    (EXPORT_DIR/"classes.json").write_text(json.dumps(CLASSES,indent=2),encoding="utf-8")
    (EXPORT_DIR/"metadata.json").write_text(json.dumps({"classes":CLASSES,"sequence_frames":SEQUENCE_FRAMES,
        "map_size":MAP_SIZE,"representation":"clip-normalized landmark map sequence",
        "model":"TimeDistributed CNN + GRU","preserves_motion":True},indent=2),encoding="utf-8")
    shutil.copy2(MODEL_ASSET,EXPORT_DIR/"hand_landmarker.task")
    complete.write_text(json.dumps({"complete":True}),encoding="utf-8")


## Evaluación final

El test se evalúa una sola vez con signantes apartados. Compara accuracy, macro F1, reporte por clase y matriz de confusión con los resultados de flujo óptico; las métricas siguen siendo preliminares por el tamaño y la distribución de signantes.

In [ ]:
X_test,y_test=stack_maps(test_df)
probs=map_model.predict(X_test,batch_size=8,verbose=0); pred=probs.argmax(axis=1)
metrics={"n_test":int(len(y_test)),"accuracy":float(accuracy_score(y_test,pred)),
    "macro_f1":float(f1_score(y_test,pred,average="macro",zero_division=0)),
    "classification_report":classification_report(y_test,pred,labels=[0,1,2],target_names=CLASSES,
        output_dict=True,zero_division=0)}
(EXPORT_DIR/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
pd.DataFrame({"video_rel":test_df.video_rel,"signer":test_df.signer,
    "true":[CLASSES[i] for i in y_test],"prediction":[CLASSES[i] for i in pred]}).to_csv(
        EXPORT_DIR/"test_predictions.csv",index=False)
cm=confusion_matrix(y_test,pred,labels=[0,1,2],normalize="true")
fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(
    ax=ax,cmap="Blues",colorbar=False,values_format=".2f")
fig.tight_layout(); fig.savefig(EXPORT_DIR/"confusion_matrix.png",dpi=150); plt.show()
print({k:v for k,v in metrics.items() if k!="classification_report"})
print("Modelo y checkpoints:",EXPORT_DIR)
